# 📋 Phase 5 — AI Audit Log
**วิชา:** Introduction to Natural Language Processing  
**โปรเจกต์:** End-to-End NLP Insight & Classification System  
**Domain:** Restaurant Reviews (Yelp.com)

---

## วัตถุประสงค์ของ AI Audit Log

AI Audit Log บันทึกทุกครั้งที่ใช้ AI (Claude) ช่วยในโปรเจกต์นี้  
โดยระบุ:
- **Task** — งานที่ขอให้ AI ช่วย
- **Prompt** — คำสั่งที่ใช้จริง (paraphrased)
- **AI Output** — สิ่งที่ AI ตอบกลับมา
- **Verification** — ผลการตรวจสอบโดยมนุษย์
- **Status** — Pass / Pass w/ Edit / Fail + Fix

> ⚠️ **หลักการสำคัญ:** AI เป็นเครื่องมือช่วย ไม่ใช่ผู้ตัดสินใจ  
> ทุกผลลัพธ์จาก AI ต้องผ่านการ verify โดยมนุษย์ก่อนใช้จริงเสมอ


In [ ]:
import pandas as pd
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_rows', 200)

print("Phase 5 — AI Audit Log")
print("=" * 50)


---
## 📦 Phase 1 — Data Engineering

งานหลัก: Web Scraping (Yelp), Text Cleaning, Tokenization/Lemmatization


In [ ]:
phase1_entries = [
    {
        "Phase": "1",
        "Task": "Regex URL Cleaning",
        "Prompt (paraphrased)": "Write regex to remove URLs from review text",
        "AI Output": "r'http\\S+'",
        "Human Verification": "Tested on sample texts — www. URLs not removed",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Extended pattern to r'http\\S+|www\\.\\S+'"
    },
    {
        "Phase": "1",
        "Task": "spaCy Tokenizer",
        "Prompt (paraphrased)": "Tokenize and lemmatize text, remove stopwords using spaCy",
        "AI Output": "Basic list comprehension with token.lemma_ and token.is_stop",
        "Human Verification": "Numbers and punctuation passed through incorrectly",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Added token.is_alpha filter to remove non-alphabetic tokens"
    },
    {
        "Phase": "1",
        "Task": "DataFrame Safety Guard",
        "Prompt (paraphrased)": "Convert scraped list to DataFrame and access label column",
        "AI Output": "pd.DataFrame(all_scraped) then df['label'] directly",
        "Human Verification": "KeyError: 'label' when list was empty",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Added empty check: if df_scraped.empty or 'label' not in df_scraped.columns"
    },
    {
        "Phase": "1",
        "Task": "แหล่งข้อมูล Web Scraping",
        "Prompt (paraphrased)": "Suggest web scraping approach for restaurant reviews",
        "AI Output": "Yelp scraping ด้วย requests + BeautifulSoup",
        "Human Verification": "Yelp ใช้ Cloudflare block — ไม่สามารถ scrape ได้จริง",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยนมาใช้ Google Maps Reviews ผ่าน Apify Scraper แทน"
    },
    {
        "Phase": "1",
        "Task": "Business IDs Hallucination",
        "Prompt (paraphrased)": "List popular restaurant identifiers for scraping",
        "AI Output": "สร้าง Yelp IDs ขึ้นมาเอง เช่น 'joes-pizza-new-york-41'",
        "Human Verification": "ID ไม่มีอยู่จริงบน Yelp — AI Hallucination",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "ใช้ Apify ที่ดึง Place ID จาก Google Maps โดยตรง ไม่ต้องระบุ ID เอง"
    },
    {
        "Phase": "1",
        "Task": "Labeling Strategy (Star → Label)",
        "Prompt (paraphrased)": "How to convert star rating to binary sentiment label?",
        "AI Output": "1-2★ = Negative, 4-5★ = Positive, 3★ = skip",
        "Human Verification": "สอดคล้องกับ literature standard สำหรับ binary sentiment",
        "Status": "✅ Pass",
        "Fix Applied": "—"
    },
    {
        "Phase": "1",
        "Task": "รวม 2 CSV ไฟล์",
        "Prompt (paraphrased)": "How to merge two DataFrames with same schema?",
        "AI Output": "pd.concat([df1, df2], ignore_index=True)",
        "Human Verification": "ทำงานถูกต้อง ตรวจสอบด้วย df.shape และ value_counts()",
        "Status": "✅ Pass",
        "Fix Applied": "—"
    },
    {
        "Phase": "1",
        "Task": "กรอง Null Text",
        "Prompt (paraphrased)": "Filter out rows with missing review text",
        "AI Output": "df.dropna(subset=['text'])",
        "Human Verification": "ทำงานถูกต้อง เพิ่ม print จำนวน rows ที่ลบเพื่อ transparency",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "เพิ่ม print statement แสดงจำนวน rows ที่ถูกกรองออกในแต่ละขั้น"
    },
    {
        "Phase": "1",
        "Task": "Stopword Removal — Column text_no_stopwords ไม่ถูกบันทึก",
        "Prompt (paraphrased)": "Apply preprocessing pipeline and save cleaned text columns for downstream use",
        "AI Output": "บันทึกเฉพาะ text_clean — ไม่มี stopword-removed column ใน output CSV",
        "Human Verification": "Phase 3 TF-IDF Top-20 features แสดง 'the'(0.048), 'and'(0.044), 'was'(0.038) — ล้วนเป็น stopwords; ไม่มี sentiment value เลย",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม remove_stopwords() และบันทึก column text_no_stopwords; Phase 3 เปลี่ยนมาใช้ column นี้ + stop_words='english' ใน TfidfVectorizer เป็น double protection"
    },
    {
        "Phase": "1",
        "Task": "Web Scraping — เปลี่ยน Source จาก SerpApi เป็น TripAdvisor CSV",
        "Prompt (paraphrased)": "Implement review scraping using SerpApi Yelp Reviews engine",
        "AI Output": "SerpApi call ด้วย engine='yelp_reviews', place_id=business_slug",
        "Human Verification": "SerpApi ต้องใช้ numeric oseid ไม่ใช่ text slug; yelp_reviews engine คืน 'no results' ทุกร้าน; OpenTable engine ก็ใช้ไม่ได้กับ TripAdvisor data",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยน source เป็น TripAdvisor จาก CSV ที่มีอยู่ (25 ร้าน, 85,831 reviews); ดึง rest_id/loc_id จาก webUrl column ด้วย regex; scraper ใช้ OverlayWidgetAjax endpoint"
    },
    {
        "Phase": "1",
        "Task": "Star Rating → Binary Label Parser",
        "Prompt (paraphrased)": "Convert star ratings (1-5) to binary sentiment labels for classification",
        "AI Output": "1-2★ = Negative(0), 4-5★ = Positive(1), 3★ = Neutral — ไม่ได้ระบุว่าจะทำอะไรกับ Neutral",
        "Human Verification": "3★ reviews มี mixed signal — ไม่ควรใช้ใน binary classification; ต้องการ skip ไม่ใช่ label เป็น Neutral",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "เปลี่ยนเป็น: 1-2★=Negative(0), 4-5★=Positive(1), 3★=drop row; เพิ่ม print จำนวนที่ถูก drop เพื่อ transparency"
    },
]

df1 = pd.DataFrame(phase1_entries)
print(f"Phase 1 — {len(df1)} audit entries")
display(HTML(df1.to_html(index=False, escape=False,
    classes='table', border=1,
    justify='left')))


---
## 🔍 Phase 2 — Unsupervised Discovery (LDA + NER + NMF)


In [ ]:
phase2_entries = [
    {
        "Phase": "2.1 LDA",
        "Task": "LDA Setup",
        "Prompt (paraphrased)": "Set up gensim LDA with coherence score to find best k",
        "AI Output": "Basic LdaModel with default dictionary, no filter_extremes",
        "Human Verification": "Too many rare/common words degrading coherence scores",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Added dictionary.filter_extremes(no_below=3, no_above=0.8)"
    },
    {
        "Phase": "2.1 LDA",
        "Task": "Coherence Score Range",
        "Prompt (paraphrased)": "Compute coherence score for k=3 to 10",
        "AI Output": "Loop k=3 to 10, plot c_v coherence",
        "Human Verification": "Range too narrow — elbow not visible",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Extended range to k=3-12 to fully show elbow"
    },
    {
        "Phase": "2.2 NER",
        "Task": "NER Pipeline Speed",
        "Prompt (paraphrased)": "Extract named entities from all reviews using spaCy",
        "AI Output": "Loop with nlp(text) per document",
        "Human Verification": "Very slow on 2,000+ reviews",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Replaced with nlp.pipe() batch processing (3-5x faster)"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "NMF Initialization",
        "Prompt (paraphrased)": "Set up NMF topic modeling with scikit-learn",
        "AI Output": "NMF with default init='random'",
        "Human Verification": "Results inconsistent across runs",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Changed to init='nndsvda' for deterministic, better initialization"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "TF-IDF Configuration",
        "Prompt (paraphrased)": "Configure TF-IDF vectorizer for NMF input",
        "AI Output": "TfidfVectorizer with unigrams only",
        "Human Verification": "Missing domain-specific bigrams like 'wait time', 'good service'",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Added ngram_range=(1,2) and sublinear_tf=True"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "Topic Diversity Metric",
        "Prompt (paraphrased)": "Suggest metric to evaluate NMF topic quality alongside reconstruction error",
        "AI Output": "Topic Diversity = unique_words / total_words from top-10 per topic",
        "Human Verification": "Score = 1.000 for every k — metric insensitive on large vocabulary dataset",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Replaced with Elbow Method (Reconstruction Error) only"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "NMF Heatmap Scale",
        "Prompt (paraphrased)": "Visualize H matrix as heatmap across topics",
        "AI Output": "Raw H values plotted with shared color scale",
        "Human Verification": "Topics with small weights appeared invisible — misleading visualization",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Normalize each topic row to max=1 before plotting"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "Topic Labels IndexError",
        "Prompt (paraphrased)": "Auto-assign topic labels from predefined list",
        "AI Output": "Direct index access topic_labels[i] without bounds check",
        "Human Verification": "IndexError when k > len(topic_labels)",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Added fallback: topic_labels[i] if i < len(topic_labels) else f'Topic {i+1}'"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "วิธีหา Optimal k",
        "Prompt (paraphrased)": "How to find optimal number of topics for NMF?",
        "AI Output": "แนะนำ Coherence Score (c_v) ด้วย Gensim",
        "Human Verification": "c_v ให้ค่าเท่ากันทุก k (0.3947) — dataset เล็กเกินไปสำหรับ sliding window co-occurrence. ลอง u_mass แล้วยังเท่ากันทุก k เช่นกัน",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "กลับมาใช้ Elbow Method (Reconstruction Error) ซึ่งทำงานได้กับทุก dataset size"
    },
    {
        "Phase": "2.3 NMF",
        "Task": "k range สำหรับ Elbow",
        "Prompt (paraphrased)": "What range of k should I test for restaurant review topics?",
        "AI Output": "แนะนำ range(3, 12)",
        "Human Verification": "range(5, 11) ทำให้ elbow อยู่นอก range — error เท่ากันทุก k อีกครั้ง",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยนเป็น range(1, 11) เพื่อให้เห็น drop curve ตั้งแต่ k=1"
    },
    {
        "Phase": "2.2 NER",
        "Task": "NER False-Positive Filtering — ORG entities",
        "Prompt (paraphrased)": "Extract and visualize top ORG named entities from restaurant reviews",
        "AI Output": "ไม่มี post-filter — accept ทุก entity ที่ spaCy tag เป็น ORG",
        "Human Verification": "Top 15 ORG chart: 'un'(18), 'never'(10), 'az'(9), 'free'(6), 'awesome'(5) — ล้วนไม่ใช่ organization; spaCy en_core_web_sm classify adjectives/adverbs ผิดบน short review text",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม is_valid_entity() function ด้วย 5 rules: (1) len ≥ 2 chars, (2) ORG_BLACKLIST 40+ คำ, (3) digits-only filter, (4) punctuation artifact filter, (5) real letter count ≥ 3"
    },
    {
        "Phase": "2.2 NER",
        "Task": "NER False-Positive Filtering — PERSON entities",
        "Prompt (paraphrased)": "Visualize top 15 PERSON named entities from restaurant reviews",
        "AI Output": "ไม่มี PERSON filter — 'vegas'(24), 'wifi'(8), 'bellagio'(8), 'eataly'(8) ปรากฏเป็น top persons",
        "Human Verification": "สถานที่ (vegas, bellagio) และแบรนด์ (eataly) ถูก tag เป็น PERSON — model ขาด context ใน short text",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม PERSON_BLACKLIST (25+ คำ), KNOWN_PLACES (30+ สถานที่ NYC/US), KNOWN_ORGS (10+ แบรนด์); Rule 6 cross-check PERSON กับทั้ง 2 sets; log rejected_log 20 รายการแรกเพื่อ debug"
    },
    {
        "Phase": "2.1 NMF",
        "Task": "Elbow Curve — k_range ไม่เริ่มจาก k=1",
        "Prompt (paraphrased)": "Plot reconstruction error vs k (number of topics) using elbow method",
        "AI Output": "k_range = list(range(3, 12)); กราฟเริ่มที่ k=3 — ไม่เห็น baseline drop",
        "Human Verification": "กราฟจริงเริ่มที่ k=5 ทำให้ curve ไม่ครบ; ไม่เห็น steep drop ที่ k=1→2; elbow detection ทำงานบน range ที่ไม่ complete",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "แยก k_range_full=range(1,11) สำหรับ visualization และ k_range_train=range(5,11) สำหรับ train จริงตามโจทย์; k=1 ใช้ init='random' เพราะ nndsvda ต้องการ n_components≥2; กราฟแบ่ง 3 โซนสี (grey=viz only, green=fast drop, red=diminishing return)"
    },
    {
        "Phase": "2.1 NMF",
        "Task": "NMF Topic Keywords — Person Names และ Noise Words ปรากฏใน Topics",
        "Prompt (paraphrased)": "Run NMF topic modeling and visualize top-10 keywords per topic",
        "AI Output": "custom_stopwords มีแค่ restaurant domain words — ไม่มี person name filter; Topic 4 dominated โดย 'kevin'(rank#1), 'kevin great', 'kevin amazing', 'kevin best'; Topic 3 มี 'pear', 'diego', 'gam'; Topic 5 มี 'ive', 'ive ever', 'far best'",
        "Human Verification": "Person names คือ reviewer identities ไม่ใช่ thematic signal; 'ive' = fragment ของ I've; 'pear','gam','diego' = ชื่อคนที่ NER ระบุไว้แล้วใน Phase 2.2; topic 4 label 'Wait Time & Reservation' ผิดทั้งหมด เพราะ keywords ล้วนเป็นชื่อ kevin",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม person_stopwords 35+ ชื่อ (kevin, diego, gam, pear, adam, ...) และ noise_stopwords (ive, far, much, thing, better, ordered, ...); เพิ่ม double-filter ใน preprocess_for_nmf(); เพิ่ม stop_words='english' ใน TfidfVectorizer; ปรับ max_df=0.85; เพิ่ม person leak check หลัง fit"
    },
    {
        "Phase": "2.1 NMF",
        "Task": "NMF Topic Keywords Round 2 — Cross-Topic Generic Words และ Topic Labels ผิด",
        "Prompt (paraphrased)": "Re-run NMF after person name filtering — visualize top-10 keywords per topic",
        "AI Output": "หลังกรอง person names ออกแล้ว ยังมี cross-topic words: 'great'(T2 rank#1 weight 1.8), 'food','service','experience','best','recommend' ปรากฏทุก topic; 'nni','erica','manhattan' ยังหลุดเข้ามา; 'best best' bigram ซ้ำ; Topic labels เก่าไม่ตรงกับ keywords จริง",
        "Human Verification": "Topic 4 label 'Wait Time & Reservation' แต่ keywords จริงคือ pizza/pasta/chef — ผิดทั้งหมด; Topic 5 label 'Value & Price' แต่ keywords คือ delicious/recommend; cross-topic words ทำให้ topics ไม่ distinct",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม cross-topic generics เข้า custom_stopwords: great, food, service, experience, best, recommend, love, good, way, bar, definitely, highly; เพิ่ม erica, nni เข้า person_stopwords; เพิ่ม manhattan เข้า noise_stopwords; อัปเดต topic_labels ให้สะท้อน keywords จริง (T4: Pizza & Italian Food, T5: Delicious Food & Recommendation)"
    },
    {
        "Phase": "2.1 NMF",
        "Task": "NMF Topic Labels — ตั้งล่วงหน้าโดยไม่ดู Keywords จริง",
        "Prompt (paraphrased)": "Assign descriptive labels to NMF topics for restaurant reviews",
        "AI Output": "ตั้ง labels ล่วงหน้า: T1=Food Quality, T2=Service, T3=Ambiance, T4=Wait Time, T5=Value, T6=Delivery — ก่อนดู keywords จริงที่ออกมา",
        "Human Verification": "หลัง run จริง: T4 label='Wait Time & Reservation' แต่ keywords คือ kevin/pizza/best; T5='Value & Price' แต่ keywords คือ delicious/recommend — labels ไม่ตรงกับ content",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "ดู top-10 keywords จริงก่อนตั้ง label; อัปเดตเป็น T4=Pizza & Italian Food, T5=Delicious Food & Recommendation; เพิ่ม comment ว่า labels ต้องตั้งหลัง run เสมอ"
    },
]

df2 = pd.DataFrame(phase2_entries)
print(f"Phase 2 — {len(df2)} audit entries")
display(HTML(df2.to_html(index=False, escape=False,
    classes='table', border=1, justify='left')))


---
## 🤖 Phase 3 — Classic vs Neural (Sentiment Classification)


In [ ]:
phase3_entries = [
    {
        "Phase": "3",
        "Task": "TF-IDF + Classical ML Setup",
        "Prompt (paraphrased)": "Set up TF-IDF with Naive Bayes, Logistic Regression, and Random Forest classifiers",
        "AI Output": "Basic TfidfVectorizer(max_features=5000) + 3 models with default params",
        "Human Verification": "Worked correctly — added ngram_range=(1,2) and max_features=10000 for richer features",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Tuned TF-IDF: ngram_range=(1,2), max_features=10000, min_df=2"
    },
    {
        "Phase": "3",
        "Task": "Word2Vec Training",
        "Prompt (paraphrased)": "Train Word2Vec on training corpus for restaurant reviews",
        "AI Output": "Word2Vec with CBOW (sg=0), vector_size=100, window=5",
        "Human Verification": "Skip-gram (sg=1) performs better for domain-specific infrequent words",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Changed to sg=1 (Skip-gram), kept vector_size=100, min_count=2"
    },
    {
        "Phase": "3",
        "Task": "LSTM Architecture",
        "Prompt (paraphrased)": "Build LSTM model for sentiment classification using Word2Vec embeddings",
        "AI Output": "Simple unidirectional LSTM(64) + Dense",
        "Human Verification": "Bidirectional LSTM captures both forward and backward context better",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Changed to Bidirectional(LSTM(64)) + Dropout(0.3) + Dense(1, sigmoid)"
    },
    {
        "Phase": "3",
        "Task": "LSTM Overfitting",
        "Prompt (paraphrased)": "Train LSTM for 20 epochs on sentiment data",
        "AI Output": "model.fit() with epochs=20, no callbacks",
        "Human Verification": "val_loss increased after epoch 5-6 — clear overfitting",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Added EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)"
    },
    {
        "Phase": "3",
        "Task": "Embedding Matrix Construction",
        "Prompt (paraphrased)": "Initialize Keras Embedding layer with Word2Vec pretrained weights",
        "AI Output": "Loop over vocab and assign w2v vectors to embedding_matrix",
        "Human Verification": "OOV words initialized to zeros — verified acceptable behavior",
        "Status": "✅ Pass",
        "Fix Applied": "—"
    },
    {
        "Phase": "3",
        "Task": "Model Evaluation Helper",
        "Prompt (paraphrased)": "Write function to evaluate classifier and return metrics dict",
        "AI Output": "evaluate_model() returning accuracy, precision, recall, F1, ROC-AUC",
        "Human Verification": "All metrics computed correctly, verified against sklearn manual calls",
        "Status": "✅ Pass",
        "Fix Applied": "—"
    },
    {
        "Phase": "3",
        "Task": "TF-IDF Input Column — ใช้ text_clean แทน text_no_stopwords",
        "Prompt (paraphrased)": "Load preprocessed data and vectorize with TF-IDF for Naive Bayes, LR, and RF",
        "AI Output": "df['text_input'] = df['text_clean'] — text ที่ยังมี stopwords; TfidfVectorizer ไม่มี stop_words parameter",
        "Human Verification": "Top 20 TF-IDF Features: 'the'(0.048), 'and'(0.044), 'was'(0.038), 'to'(0.032) — ล้วนเป็น stopwords; ไม่มี sentiment word ใน top features เลย",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยนเป็น df['text_input'] = df['text_no_stopwords'] (Layer 1); เพิ่ม stop_words='english' ใน TfidfVectorizer (Layer 2 — กัน stopwords จาก n-gram edge); top features เปลี่ยนเป็น 'delicious', 'terrible', 'amazing'"
    },
]

df3 = pd.DataFrame(phase3_entries)
print(f"Phase 3 — {len(df3)} audit entries")
display(HTML(df3.to_html(index=False, escape=False,
    classes='table', border=1, justify='left')))


---
## 📊 Phase 4 — Evaluation & Semantic Search


In [ ]:
phase4_entries = [
    {
        "Phase": "4",
        "Task": "Full Evaluation Dashboard",
        "Prompt (paraphrased)": "Create comprehensive evaluation comparing all models with per-class metrics",
        "AI Output": "Single summary table with macro averages only",
        "Human Verification": "Missing per-class breakdown — important for imbalanced data",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Extended to show Precision/Recall/F1 separately for Positive and Negative classes"
    },
    {
        "Phase": "4",
        "Task": "Learning Curve",
        "Prompt (paraphrased)": "Plot learning curve to detect overfitting for classical models",
        "AI Output": "sklearn learning_curve() with 5 train sizes",
        "Human Verification": "Too few points — curve too coarse to read trend",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Increased to 8 train sizes with np.linspace(0.1, 1.0, 8)"
    },
    {
        "Phase": "4",
        "Task": "Document Vector Encoding",
        "Prompt (paraphrased)": "Encode documents into vectors using Word2Vec for semantic search",
        "AI Output": "get_doc_vector() with mean pooling of word vectors",
        "Human Verification": "OOV documents returned zero vectors — could cause misleading similarity",
        "Status": "⚠️ Pass w/ Edit",
        "Fix Applied": "Added zero vector guard: filtered out zero vectors before similarity search"
    },
    {
        "Phase": "4",
        "Task": "t-SNE Parameter (n_iter → max_iter)",
        "Prompt (paraphrased)": "Visualize document embedding space with t-SNE 2D projection",
        "AI Output": "TSNE(n_components=2, perplexity=30, n_iter=1000)",
        "Human Verification": "TypeError: TSNE got unexpected keyword argument 'n_iter' (sklearn >= 1.2 renamed it)",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Changed n_iter=1000 to max_iter=1000"
    },
    {
        "Phase": "4",
        "Task": "Sentiment Distribution Chart — Language",
        "Prompt (paraphrased)": "Add subtitle explaining the sentiment distribution chart",
        "AI Output": "Thai subtitle: 'Query บวก ควรดึง Positive reviews มากกว่า...'",
        "Human Verification": "Notebook should be in English for academic submission",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Changed to English: 'Positive queries should retrieve more Positive reviews, and vice versa'"
    },
    {
        "Phase": "4",
        "Task": "Feature Ablation Study",
        "Prompt (paraphrased)": "Compare model performance across different feature engineering approaches",
        "AI Output": "Bar chart comparing TF-IDF unigram vs bigram vs Word2Vec vs LSTM",
        "Human Verification": "Correct structure — verified feature configs match actual trained models",
        "Status": "✅ Pass",
        "Fix Applied": "—"
    },
    {
        "Phase": "4",
        "Task": "Document Search Backend — Word2Vec → TF-IDF Cosine Similarity",
        "Prompt (paraphrased)": "Build document search using Word2Vec mean pooling; visualize sentiment distribution of top-20 results per query",
        "AI Output": "Simple mean pooling ของ Word2Vec vectors — 'terrible service rude staff' คืน 19 Positive / 1 Negative; 'great value cheap price' คืน 19 Positive / 1 Negative",
        "Human Verification": "ทุก query dominated โดย Positive ไม่ว่า query จะเป็น positive หรือ negative — Word2Vec train บน corpus เล็ก ~2,000 reviews ทำให้ 'terrible' ≈ 'delicious' ใน vector space; Phase 3 พิสูจน์แล้วว่า TF-IDF ดีกว่า (NB F1=0.89 vs LSTM F1=0.79)",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยน backend เป็น TF-IDF cosine similarity (ngram=(1,2), sublinear_tf=True, stop_words='english'); อัปเดต t-SNE ให้ใช้ TF-IDF sparse matrix; เพิ่ม QUERY_EXPECT dict และ ✅/❌ match indicator ใน output"
    },
    {
        "Phase": "4",
        "Task": "Search Query Design — Ambiguous Words",
        "Prompt (paraphrased)": "Design 4 test queries to evaluate semantic search covering different sentiment categories",
        "AI Output": "Q3: 'long wait time slow', Q4: 'great value cheap price' — คำที่ Positive reviews ใช้ด้วย: 'worth the wait', 'good value for money'",
        "Human Verification": "Q3 ได้ 7 Pos / 13 Neg (ดีขึ้น), Q4 ยังได้ 19 Pos / 1 Neg — คำ 'great', 'value', 'cheap', 'price' ล้วนอยู่ใน Positive reviews เยอะมาก",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Q3 เปลี่ยนเป็น 'waited forever ignored never seated nightmare rude'; Q4 เปลี่ยนเป็น 'inedible disgusting filthy overpriced worst regret' — ใช้คำที่แทบไม่ปรากฏใน Positive reviews; เพิ่ม Query Design comment อธิบาย principle"
    },
    {
        "Phase": "4",
        "Task": "NB Log Probability Ratio — ชื่อคนเป็น Top Positive Features",
        "Prompt (paraphrased)": "Visualize Naive Bayes most discriminative words using log probability ratio (feature_log_prob_ diff)",
        "AI Output": "ไม่มี name/noise filter — 'kevin'(3.04), 'kevin was'(2.74), 'adam'(2.12), 'pear'(2.07) อยู่ใน top 20 Positive Indicators",
        "Human Verification": "ชื่อคนไม่ใช่ sentiment discriminator — 'kevin' rank #1 เพราะ reviewer ชื่อ Kevin เขียน positive reviews มาก; ไม่ใช่เพราะชื่อมี sentiment; เป็น data leakage จาก reviewer identity",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เพิ่ม PERSON_FILTER 30+ ชื่อ (อ้างอิงจาก NER Phase 2.2); is_sentiment_word() กรองทั้ง unigram ชื่อคน และ bigram ที่ขึ้นต้นด้วยชื่อคน (เช่น 'kevin was', 'adam said'); top features แสดงเฉพาะ genuine sentiment words"
    },
    {
        "Phase": "4",
        "Task": "Semantic Search Sentiment Chart — Iteration 1 (Word2Vec baseline)",
        "Prompt (paraphrased)": "Visualize sentiment distribution of top-20 search results per query using Word2Vec",
        "AI Output": "กราฟแรก: 'delicious food amazing'=20Pos/0Neg ✅, 'terrible service rude'=19Pos/1Neg ❌, 'long wait time slow'=19Pos/1Neg ❌, 'great value cheap price'=19Pos/1Neg ❌",
        "Human Verification": "3 ใน 4 query ผิดทั้งหมด — Word2Vec simple mean pooling บน corpus เล็กทำให้ sentiment vector เหมือนกัน; เปลี่ยน backend เป็น TF-IDF",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "เปลี่ยนเป็น TF-IDF cosine similarity; ผล Iteration 2: Q2='terrible horrible rude'=1Pos/19Neg ✅"
    },
    {
        "Phase": "4",
        "Task": "Semantic Search Sentiment Chart — Iteration 2 (TF-IDF, queries v1)",
        "Prompt (paraphrased)": "Re-run sentiment distribution chart with TF-IDF backend and original queries",
        "AI Output": "กราฟที่ 2: Q1=20Pos/0Neg ✅, Q2=1Pos/19Neg ✅, Q3(long wait time slow)=7Pos/13Neg ⚠️, Q4(great value cheap price)=19Pos/1Neg ❌",
        "Human Verification": "Q3 ดีขึ้นแต่ยังไม่ dominant; Q4 ยังผิด — 'great','value','price' อยู่ใน Positive reviews เยอะมาก เช่น 'great value for money'",
        "Status": "❌ Fail → Fixed",
        "Fix Applied": "Q3 เปลี่ยนเป็น 'waited forever ignored never seated nightmare rude'; Q4 เปลี่ยนเป็น 'inedible disgusting filthy overpriced worst regret'"
    },
]

df4 = pd.DataFrame(phase4_entries)
print(f"Phase 4 — {len(df4)} audit entries")
display(HTML(df4.to_html(index=False, escape=False,
    classes='table', border=1, justify='left')))


---
## 📈 Summary Dashboard — AI Audit Statistics


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

all_entries = phase1_entries + phase2_entries + phase3_entries + phase4_entries
df_all = pd.DataFrame(all_entries)

# สรุปสถิติ
status_map = {
    '✅ Pass':          'Pass',
    '⚠️ Pass w/ Edit': 'Pass w/ Edit',
    '❌ Fail → Fixed':  'Fail → Fixed',
}
df_all['Status_clean'] = df_all['Status'].map(status_map).fillna(df_all['Status'])

status_counts  = df_all['Status_clean'].value_counts()
phase_counts   = df_all['Phase'].apply(lambda x: 'Phase ' + str(x).split('.')[0]).value_counts().sort_index()

total  = len(df_all)
n_pass = (df_all['Status_clean'] == 'Pass').sum()
n_edit = (df_all['Status_clean'] == 'Pass w/ Edit').sum()
n_fail = (df_all['Status_clean'] == 'Fail → Fixed').sum()

print(f"{'='*55}")
print(f"  TOTAL AI INTERACTIONS : {total:>3}")
print(f"  ✅ Pass (no changes)  : {n_pass:>3}  ({n_pass/total*100:.0f}%)")
print(f"  ⚠️  Pass w/ Edit      : {n_edit:>3}  ({n_edit/total*100:.0f}%)")
print(f"  ❌ Fail → Fixed       : {n_fail:>3}  ({n_fail/total*100:.0f}%)")
print(f"{'='*55}")
print(f"  Human modification rate : {(n_edit+n_fail)/total*100:.0f}% of all AI outputs")
print(f"  This confirms AI-assisted ≠ AI-automated")

# ── กราฟ ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('AI Audit Log — Summary Dashboard', fontsize=14, fontweight='bold')
fig.patch.set_facecolor('#f8f9fa')

STATUS_COLORS = {
    'Pass':           '#2ecc71',
    'Pass w/ Edit':   '#f39c12',
    'Fail → Fixed':   '#e74c3c',
}

# ─ Pie: Overall Status ─────────────────────────────────────────────────
ax = axes[0]
pie_labels = [f"{k}\n({v})" for k, v in status_counts.items()]
pie_colors = [STATUS_COLORS.get(k, '#95a5a6') for k in status_counts.index]
wedges, texts, autotexts = ax.pie(
    status_counts.values,
    labels=pie_labels,
    colors=pie_colors,
    autopct='%1.0f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2),
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax.set_title('Overall AI Output Status', fontsize=12, fontweight='bold')

# ─ Bar: Status per Phase ───────────────────────────────────────────────
ax = axes[1]
phases = sorted(df_all['Phase'].apply(lambda x: 'Phase ' + str(x).split('.')[0]).unique())
bar_data = {s: [] for s in ['Pass', 'Pass w/ Edit', 'Fail → Fixed']}
for ph in phases:
    sub = df_all[df_all['Phase'].apply(lambda x: 'Phase ' + str(x).split('.')[0]) == ph]
    for s in bar_data:
        bar_data[s].append((sub['Status_clean'] == s).sum())

x = np.arange(len(phases))
width = 0.25
for i, (status, vals) in enumerate(bar_data.items()):
    bars = ax.bar(x + i * width, vals, width,
                  label=status, color=STATUS_COLORS[status],
                  alpha=0.85, edgecolor='white')
    ax.bar_label(bars, padding=2, fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(phases, fontsize=10)
ax.set_ylabel('Number of Tasks', fontsize=11)
ax.set_title('AI Output Status per Phase', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.set_facecolor('#f8f9fa')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

# ─ Horizontal Bar: Tasks per Phase ────────────────────────────────────
ax = axes[2]
phase_cnt = df_all['Phase'].apply(lambda x: 'Phase ' + str(x).split('.')[0]).value_counts().sort_index()
bar_cols   = ['#3498db','#9b59b6','#1abc9c','#e67e22']
hbars = ax.barh(phase_cnt.index, phase_cnt.values,
                color=bar_cols[:len(phase_cnt)], alpha=0.85,
                height=0.55, edgecolor='white')
ax.bar_label(hbars, padding=4, fontsize=11, fontweight='bold')
ax.set_xlabel('Number of AI Interactions', fontsize=11)
ax.set_title('Total AI Interactions per Phase', fontsize=12, fontweight='bold')
ax.set_facecolor('#f8f9fa')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, max(phase_cnt.values) * 1.2)

plt.tight_layout()
plt.savefig('phase5_audit_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phase5_audit_summary.png')


---
## 🔑 Key Findings from AI Audit

### ประเภทของ Bug ที่พบบ่อยที่สุด

| ประเภท Bug | ตัวอย่าง | จำนวน |
|---|---|:---:|
| **Architecture/Config ที่ default ไม่ดีพอ** | CBOW→Skip-gram, random→nndsvda, no early stopping | 4 |
| **Edge Case ไม่ครอบคลุม** | Empty DataFrame, OOV words, zero vectors | 3 |
| **Visualization ที่อ่านผิดได้** | Heatmap ไม่ normalize, Elbow เริ่ม k=3, Thai label | 3 |
| **NMF Stopword / Noise Word หลุดเข้า Topics** | Person names, cross-topic generics, bigram artifacts | 3 |
| **NER False-Positive (Entity Misclassification)** | ORG: 'un','never'; PERSON: 'vegas','eataly' | 2 |
| **Wrong Column / Missing Preprocessing Step** | text_clean แทน text_no_stopwords | 2 |
| **Metric ที่ไม่เหมาะกับ Dataset** | Topic Diversity = 1.000 ทุก k; Coherence เท่ากันทุก k | 2 |
| **Topic Labels ตั้งก่อนดู Keywords จริง** | T4='Wait Time' แต่ keywords คือ pizza/pasta | 2 |
| **Search Query / Feature Design** | Ambiguous query words; ชื่อคนเป็น top NB feature | 2 |
| **Iterative Search Result Fix** | TF-IDF queries ต้อง iterate 2 รอบจึงได้ผลถูกต้อง | 2 |
| **Wrong Data Source / API Format** | SerpApi slug vs numeric oseid | 1 |
| **Wrong ML Method for Dataset Size** | Word2Vec search บน corpus ~2,000 reviews | 1 |
| **API/Library Version Change** | `n_iter` → `max_iter` ใน sklearn ≥1.2 | 1 |
| **AI Hallucination** | Yelp Business IDs ที่สร้างขึ้นมาเอง | 1 |
| **Total** | | **29** |

---

### สถิติ AI Audit ภาพรวม

| Phase | Total Entries | ✅ Pass | ⚠️ Pass w/ Edit | ❌ Fail → Fixed |
|---|:---:|:---:|:---:|:---:|
| Phase 1 — Data Engineering | 11 | 2 | 3 | 6 |
| Phase 2 — Topic Modeling (LDA/NER/NMF) | 16 | 0 | 4 | 12 |
| Phase 3 — Classic vs Neural | 7 | 2 | 3 | 2 |
| Phase 4 — Evaluation & Search | 11 | 1 | 3 | 7 |
| **รวม** | **45** | **5** | **13** | **27** |

→ **Human modification rate: 89%** (40/45 entries ต้องแก้ไขหรือปรับ)

---

### บทเรียนสำคัญ (Lessons Learned)

1. **AI ไม่รู้ version ของ library ที่ใช้จริง** — parameter อาจ deprecated ต้อง cross-check กับ docs เสมอ

2. **AI คิด happy path เป็นหลัก** — edge cases เช่น empty data, OOV, blocked scraping ต้องเพิ่มเองเสมอ

3. **AI Hallucination เกิดจริง** — Yelp Business IDs ที่คิดขึ้นมาเองไม่มีอยู่จริง; ข้อมูล real-world ต้อง verify เสมอ

4. **Default parameters ≠ best parameters** — `init='random'`, `sg=0`, unidirectional LSTM ล้วน suboptimal สำหรับ use case นี้

5. **AI ไม่รู้ domain context ของ feature** — ชื่อคน (kevin, erica) ถูก rank เป็น top NMF topic และ NB feature โดยไม่กรอง entity type

6. **Small corpus เปลี่ยน best practice** — Word2Vec ต้องการ corpus ใหญ่; บน ~2,000 reviews TF-IDF ดีกว่าทุก metric

7. **Visualization ต้องคิดถึงผู้อ่าน** — AI สร้างกราฟที่ technically ถูกแต่อ่านผิดได้ เช่น scale ไม่ normalize, elbow ไม่เริ่ม k=1

8. **Topic Labels ต้องตั้งหลัง run** — การตั้ง label ล่วงหน้าก่อนดู keywords จริงทำให้ label ไม่ตรง; ต้อง iterate เสมอ

9. **NMF Stopword Filtering ต้อง Iterate** — ต้องรัน → ดูกราฟ → เพิ่ม stopwords → รันใหม่ อย่างน้อย 2-3 รอบ; AI ไม่รู้ล่วงหน้าว่าคำไหนจะ dominate

---

### สรุป: บทบาทที่เหมาะสมของ AI ในโปรเจกต์นี้

| บทบาทที่ AI เหมาะสม ✅ | บทบาทที่ต้องระวัง ⚠️ |
|---|---|
| เขียน boilerplate code ได้เร็ว | ค้นหา real-world data (IDs, URLs) |
| แนะนำ library และ API usage | เลือก hyperparameters โดยไม่มี domain context |
| สร้าง visualization structure | รับประกัน library version compatibility |
| อธิบาย algorithm และ theory | Handle edge cases โดยอัตโนมัติ |
| Debug เมื่อ error message ชัดเจน | ตั้ง Topic Labels โดยไม่ดู output จริง |
| เสนอ algorithm alternatives | กรอง noise ออกจาก NLP features (ชื่อคน, entity) |
| สร้าง code framework ได้เร็ว | เลือก method ที่เหมาะกับ dataset size จริงๆ |


---
## 📄 Complete Audit Log (All Phases)


In [ ]:
print(f"Complete AI Audit Log — {len(df_all)} total entries")
print("=" * 60)

# แสดงตามลำดับ phase
for phase_prefix in ['1', '2', '3', '4']:
    sub = df_all[df_all['Phase'].str.startswith(phase_prefix)]
    if len(sub) == 0:
        continue
    pass_c = (sub['Status_clean'] == 'Pass').sum()
    edit_c = (sub['Status_clean'] == 'Pass w/ Edit').sum()
    fail_c = (sub['Status_clean'] == 'Fail → Fixed').sum()
    print(f"\nPhase {phase_prefix}: {len(sub)} tasks  "
          f"[✅ {pass_c}  ⚠️ {edit_c}  ❌ {fail_c}]")

print(f"\nGrand Total: {len(df_all)} tasks")
print(f"  Human modification rate: {(n_edit+n_fail)/total*100:.0f}% "
      f"({n_edit+n_fail}/{total} tasks required human intervention)")

# Export
df_all.to_csv('phase5_ai_audit_log.csv', index=False, encoding='utf-8-sig')
print("\nSaved: phase5_ai_audit_log.csv")

display(HTML(df_all[['Phase','Task','AI Output','Status','Fix Applied']]
    .to_html(index=False, escape=False, classes='table', border=1, justify='left')))
